In [312]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn import set_config
from sklearn.model_selection import train_test_split

from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv

# Random Surivial Forests

- an ensemble of survival trees
- survival = survival event + time object
- uses the log-rank test for feature & threshold selection

In [313]:
# load datasets
expr = pd.read_csv("../datasets/csv_files/deg_expression_matrix_train.csv")
clinical = pd.read_csv("../datasets/csv_files/clinical_metadata_train.csv")
expr_test = pd.read_csv("../datasets/csv_files/expression_matrix_test_one.csv")
clinical_test = pd.read_csv("../datasets/csv_files/clinical_metadata_test_one.csv")
expr.head()

,sample_name,ACO1,ADAR,ADIPOQ,AGPAT2,ALDOC,ANG,ANGPT1,ANXA1,AP1M2,...,UBE2S,VAMP3,VAMP8,VAV3,VIM,VSIG4,VTCN1,WASF3,XG,ZBTB16
0,GSM1045191,6.697765,9.719198,11.876584,8.604980,8.414169,7.941133,6.564200,9.817748,5.556355,...,4.445544,8.155041,7.942988,4.577871,10.001357,7.563904,6.866327,9.021858,4.897770,8.081731
1,GSM1045192,5.872591,9.678302,7.153605,6.503194,4.581926,4.814703,3.905613,7.074081,5.120402,...,4.422642,7.358038,9.970144,4.369697,8.650840,7.693408,4.912463,4.480019,3.334647,4.078880
2,GSM1045193,4.704279,10.457587,5.392332,6.142293,6.235045,4.186915,3.946060,4.031208,4.889919,...,4.878597,6.282494,8.971080,4.458827,7.124404,5.279563,4.479120,4.514484,3.458176,4.305987
3,GSM1045194,4.650540,9.122421,5.247072,6.581231,6.079837,5.460042,4.049906,6.102090,6.759978,...,4.498002,6.274657,8.636839,5.733220,7.628729,4.990067,7.731202,5.544438,3.338516,5.512143
4,GSM1045195,8.647494,10.151534,12.136559,9.370264,8.294877,8.182388,6.256205,10.263854,4.763354,...,4.847608,8.498974,7.105619,4.054034,10.257068,7.878502,6.027630,8.935739,6.005618,7.841789


In [314]:
# drop non-tumor samples from datasets
samples_to_keep = list(clinical[clinical["is_tumor"] == 1]['sample_name'])
expr_filtered = expr[expr['sample_name'].isin(samples_to_keep)]
clinical_filtered = clinical[clinical["is_tumor"] == 1]

# drop non-tumor samples from datasets
cols_to_keep_test = list(clinical_test[clinical_test["is_tumor"] == 1]['sample_name'])
expr_test_filtered = expr_test[['gene_symbol'] + cols_to_keep_test]
clinical_test_filtered = clinical_test[clinical_test["is_tumor"] == 1]
expr_test_filtered.head()

,gene_symbol,GSM540108,GSM540109,GSM540110,GSM540111,GSM540112,GSM540113,GSM540114,GSM540115,GSM540116,...,GSM540364,GSM540365,GSM540366,GSM540367,GSM540368,GSM540369,GSM540370,GSM540371,GSM540372,GSM540373
0,A1BG,6.146570,5.632868,7.407916,5.717083,5.667256,5.821602,5.582540,5.678218,6.570997,...,5.768699,5.915461,5.332402,5.880671,5.785042,6.160731,5.855461,5.794792,5.933755,5.574898
1,A1BG-AS1,5.192952,6.270687,5.594629,4.476920,5.047257,5.420414,4.874277,4.710615,5.320223,...,4.654715,4.821310,4.533131,4.630453,4.559740,4.400831,4.790866,4.519228,4.606000,4.458904
2,A1CF,4.136334,4.220885,4.546149,3.887450,3.876345,3.737222,3.619344,3.747788,3.939471,...,3.730873,3.814982,3.800488,3.891974,3.807973,3.815930,3.925795,3.902201,4.093958,3.588490
3,A2M,7.202705,6.328396,8.172368,8.021876,7.659437,8.497944,8.218462,8.664218,8.092863,...,8.240868,8.781016,8.344433,7.989748,9.159850,8.391154,7.977827,8.225967,8.341430,8.060287
4,A2M-AS1,5.378281,4.429702,6.103873,5.039355,4.568131,5.923578,5.995361,6.468561,4.866476,...,6.963635,5.482779,4.767218,5.862145,5.259731,4.957411,4.985096,5.038558,4.602791,6.741662


In [315]:
expr_test_t = expr_test_filtered.set_index('gene_symbol').T
expr_test_t.index.name = 'sample_name'
expr_test_t.reset_index(inplace=True)
expr_test_t.columns.name = None
expr_test_t.head()

,sample_name,A1BG,A1BG-AS1,A1CF,A2M,A2M-AS1,A2ML1,A2MP1,A4GALT,A4GNT,...,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3,mir-223
0,GSM540108,6.146570,5.192952,4.136334,7.202705,5.378281,4.452781,4.456638,7.204383,4.268921,...,7.207004,4.956326,4.257066,7.258950,7.369622,6.635957,8.039407,6.266267,6.821724,4.431420
1,GSM540109,5.632868,6.270687,4.220885,6.328396,4.429702,4.428543,3.932945,7.465655,3.955844,...,8.685472,4.814433,4.688268,6.475279,8.280853,6.957342,8.101518,5.852577,6.055815,4.392363
2,GSM540110,7.407916,5.594629,4.546149,8.172368,6.103873,4.444036,4.321941,7.247893,4.009840,...,9.418793,4.147277,4.013604,6.496174,5.774433,7.299554,8.195125,6.125562,6.990111,4.064667
3,GSM540111,5.717083,4.476920,3.887450,8.021876,5.039355,3.829404,3.530977,6.898759,3.860235,...,8.964749,5.658448,5.376926,6.416896,6.638156,7.852998,9.684551,5.897437,7.499180,4.209039
4,GSM540112,5.667256,5.047257,3.876345,7.659437,4.568131,3.678200,4.226887,6.474347,3.756299,...,9.051191,5.646992,5.464178,6.537697,5.560518,7.861657,8.696353,5.889624,7.680644,3.751116


In [316]:
# add relapse free survival time & event to training data
gene_cols = [col for col in expr.columns if col != 'sample_name']

train_data = pd.concat([
        expr['sample_name'],
        clinical_filtered[['relapse_free_time', 'relapse_free_event']].astype(int),
        expr[gene_cols]
    ], axis=1)

# drop null RFS values & rename columns
train_data.dropna(inplace=True)
train_data[['relapse_free_time', 'relapse_free_event']] = train_data[['relapse_free_time', 'relapse_free_event']].astype(int)

# get gene names
gene_names = list(train_data.columns[3:])

train_data.head()

,sample_name,relapse_free_time,relapse_free_event,ACO1,ADAR,ADIPOQ,AGPAT2,ALDOC,ANG,ANGPT1,...,UBE2S,VAMP3,VAMP8,VAV3,VIM,VSIG4,VTCN1,WASF3,XG,ZBTB16
17,GSM1045208,3026,0,4.799942,11.395017,6.075966,6.143011,5.383726,5.904613,3.628884,...,4.844556,6.401260,9.204786,5.647802,6.964493,4.966453,7.220646,5.180080,3.151567,3.705057
18,GSM1045209,755,1,4.971609,10.864474,8.177897,7.368229,5.854996,6.582208,3.702770,...,6.627002,6.492745,8.080794,4.995820,7.585698,5.970720,5.437708,4.532965,3.383118,4.551614
19,GSM1045210,3014,0,4.555494,10.942121,6.828506,6.252349,5.317867,5.791332,3.630954,...,4.907463,6.269060,8.533085,5.194826,6.922194,5.289430,6.854089,4.636144,3.299654,4.031339
20,GSM1045211,406,1,4.668031,10.776268,5.196359,6.861987,6.408760,4.921507,3.693400,...,5.012260,6.170261,6.865168,5.039221,7.565630,5.307223,5.566723,4.642180,3.480787,4.412832
21,GSM1045212,2225,0,5.458046,11.743277,3.567575,6.902541,8.048325,5.603227,3.580604,...,7.380517,7.418157,9.662955,5.193608,8.330933,8.333481,10.065004,5.678931,3.197258,3.402715


## Training

In [317]:
X_train = train_data[gene_names]
y_train = Surv.from_dataframe('relapse_free_event', 'relapse_free_time', train_data)

In [318]:
random_state = 20
rsf = RandomSurvivalForest(
    n_estimators=1000,
    min_samples_split=25,
    min_samples_leaf=30,
    max_features="sqrt",
    oob_score=True,
    n_jobs=-1,
    random_state=random_state
)
rsf.fit(X_train, y_train)

,n_estimators,1000
,max_depth,None
,min_samples_split,25
,min_samples_leaf,30
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,bootstrap,True
,oob_score,True
,n_jobs,-1
,random_state,20


## Testing

In [319]:
# add relapse free survival time & event to testing data
gene_cols = [col for col in expr_test_t.columns if col != 'sample_name']

test_data = pd.concat([
        expr_test_t['sample_name'],
        clinical_test_filtered[['relapse_free_time', 'relapse_free_event']],
        expr_test_t[gene_cols]
    ], axis=1)

# drop null RFS values & rename columns
test_data.dropna(inplace=True)
test_data[['relapse_free_time', 'relapse_free_event']] = test_data[['relapse_free_time', 'relapse_free_event']].astype(int)

# get gene names
gene_names = list(test_data.columns[3:])

test_data.head()

,sample_name,relapse_free_time,relapse_free_event,A1BG,A1BG-AS1,A1CF,A2M,A2M-AS1,A2ML1,A2MP1,...,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3,mir-223
0,GSM540108,731,1,6.146570,5.192952,4.136334,7.202705,5.378281,4.452781,4.456638,...,7.207004,4.956326,4.257066,7.258950,7.369622,6.635957,8.039407,6.266267,6.821724,4.431420
3,GSM540111,1770,1,5.717083,4.476920,3.887450,8.021876,5.039355,3.829404,3.530977,...,8.964749,5.658448,5.376926,6.416896,6.638156,7.852998,9.684551,5.897437,7.499180,4.209039
4,GSM540112,871,1,5.667256,5.047257,3.876345,7.659437,4.568131,3.678200,4.226887,...,9.051191,5.646992,5.464178,6.537697,5.560518,7.861657,8.696353,5.889624,7.680644,3.751116
5,GSM540113,301,1,5.821602,5.420414,3.737222,8.497944,5.923578,3.882873,3.602626,...,7.826577,5.197774,4.890690,6.444662,4.587187,8.174834,10.332553,6.433840,6.908704,4.178111
6,GSM540114,226,1,5.582540,4.874277,3.619344,8.218462,5.995361,3.741898,3.622146,...,9.342628,5.664366,5.156260,6.669061,5.831567,7.816731,9.150694,5.927160,7.267744,4.591735


In [320]:
X_test = test_data[gene_names]
X_test = X_test[X_train.columns]
y_test = Surv.from_dataframe('relapse_free_event', 'relapse_free_time', test_data)

In [321]:
c_index_train = rsf.score(X_train, y_train)
c_index_test = rsf.score(X_test, y_test)

print(f"Train C-index: {c_index_train:.5f}")
print(f"Test C-index: {c_index_test:.5f}")
print(f"OOB C-index: {rsf.oob_score_:.5f}")

Train C-index: 0.79939
Test C-index: 0.63794
OOB C-index: 0.63383
